# Week 64, Day 6: Forge: Week 64 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w064/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
print("1.", 2000 ** 2, "entries,", round(2000 ** 2 * 8 / 1e6, 1), "MB")
print("2.", 4 + 1)
print("3.", round(float(np.exp(-0.5 * 3 ** 2)), 5))

### Exercise: Part A: evidence-optimised receptive field

Complete the evidence (log marginal likelihood) for the linear-Gaussian model with prior covariance K(ell, sf2) and noise variance sn2, maximise it over the three hyperparameters, and compare the posterior mean filter with ridge regression tuned by held-out error.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import cho_factor, cho_solve
from scipy.optimize import minimize
gA = np.random.default_rng(646)
D = 40
pix = np.arange(D)
w_true = np.exp(-0.5 * ((pix - 14) / 2.5) ** 2) - 0.6 * np.exp(-0.5 * ((pix - 22) / 3.5) ** 2)
S = gA.normal(size=(200, D))                                # white-noise stimuli
r = S @ w_true + 0.5 * gA.normal(size=200)

def K_of(ell, sf2):
    return sf2 * np.exp(-0.5 * ((pix[:, None] - pix[None, :]) / ell) ** 2) + 1e-8 * np.eye(D)

def neg_evidence(lp):
    ell, sf2, sn2 = np.exp(lp)
    Cov = S @ K_of(ell, sf2) @ S.T + sn2 * np.eye(len(r))
    c, low = cho_factor(Cov, lower=True)
    return ...

res = min((minimize(neg_evidence, np.array(s0), method="L-BFGS-B") for s0 in ([1.0, 0.0, 0.0], [0.0, -1.0, -1.0], [2.0, 1.0, 0.0])), key=lambda q: q.fun)
ell_hat, sf2_hat, sn2_hat = np.exp(res.x)
Kw = K_of(ell_hat, sf2_hat)
w_gp = Kw @ S.T @ np.linalg.solve(S @ Kw @ S.T + sn2_hat * np.eye(len(r)), r)
err_gp = np.linalg.norm(w_gp - w_true)

tr, te = np.arange(150), np.arange(150, 200)
lams = np.logspace(-2, 3, 40)
ridge = [np.linalg.solve(S[tr].T @ S[tr] + lam * np.eye(D), S[tr].T @ r[tr]) for lam in lams]
val = [np.mean((r[te] - S[te] @ w_) ** 2) for w_ in ridge]
w_ridge = np.linalg.solve(S.T @ S + lams[int(np.argmin(val))] * np.eye(D), S.T @ r)
err_ridge = np.linalg.norm(w_ridge - w_true)
print(f"learned length-scale {ell_hat:.2f} pixels, noise sd {np.sqrt(sn2_hat):.2f} (true 0.5)")
print(f"error of the filter estimate: GP prior {err_gp:.3f}, best ridge {err_ridge:.3f}")
plt.plot(w_true, "k", label="true"); plt.plot(w_gp, label="GP prior"); plt.plot(w_ridge, alpha=0.7, label="ridge"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_beats_ridge():
    "the GP-prior estimate is closer to the true filter than the best ridge estimate"
    assert err_gp < err_ridge
def test_learned():
    "the learned smoothness length-scale is within a factor of 2 of the true 2.5 pixels"
    assert 1.25 < ell_hat < 5.0

test_beats_ridge()
test_learned()
print("All checks passed")

### Exercise: Part B: GP tuning curve for spike counts

Complete the Newton step for the Laplace approximation of a Poisson GP: the gradient is y - exp(f) - K^-1 f and the Hessian is -(diag(exp(f)) + K^-1). Fit a tuning curve to counts at 24 orientations sampled unevenly, and compute posterior standard deviations in a dense and a sparse region.

In [ ]:
gB = np.random.default_rng(6461)
th_all = np.r_[np.linspace(0, 1.6, 18), np.linspace(1.9, 3.1, 6)]     # dense below 1.6 rad, sparse above
rate = lambda a: 10 * np.exp(1.5 * (np.cos(2 * (a - 0.8)) - 1)) + 0.5
yB = gB.poisson(rate(th_all))
Kth = 4.0 * np.exp(-2 * np.sin(np.abs(th_all[:, None] - th_all[None, :])) ** 2 / 0.6 ** 2) + 1e-6 * np.eye(len(th_all))
Kinv = np.linalg.inv(Kth)
mu0 = np.log(yB.mean() + 1e-6)

fhat = np.zeros(len(th_all))
for it in range(50):
    lam = np.exp(fhat + mu0)
    grad = ...
    Hess = -(np.diag(lam) + Kinv)
    fhat = fhat - np.linalg.solve(Hess, grad)
W = np.diag(np.exp(fhat + mu0))
post_cov = np.linalg.inv(W + Kinv)
sd_f = np.sqrt(np.diag(post_cov))
r_fit = np.corrcoef(np.exp(fhat + mu0), rate(th_all))[0, 1]
sd_dense = sd_f[:18].mean(); sd_sparse = sd_f[18:].mean()
print(f"correlation of the fitted rate with the truth {r_fit:.3f}")
print(f"posterior sd of log rate: dense region {sd_dense:.3f}, sparse region {sd_sparse:.3f}")
plt.plot(th_all, yB, "k.", label="counts"); plt.plot(th_all, rate(th_all), "k--", label="true rate")
plt.plot(th_all, np.exp(fhat + mu0), label="GP fit"); plt.legend(); plt.xlabel("orientation (rad)"); plt.show()

In [ ]:
import numpy as np
def test_fit():
    "the fitted rate curve correlates with the true rate above 0.9"
    assert r_fit > 0.9
def test_bands():
    "posterior bands are wider where fewer stimuli were presented"
    assert sd_sparse > sd_dense

test_fit()
test_bands()
print("All checks passed")